In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

red_wine = pd.read_csv('winequality-red.csv', sep=';')
white_wine = pd.read_csv('winequality-white.csv', sep=';')
combined_wine = pd.concat([red_wine, white_wine], ignore_index=True)

datasets = {
    'Red Wine': red_wine,
    'White Wine': white_wine,
    'Combined Wine': combined_wine
}

splits = {}

for name, df in datasets.items():
    X = df.drop('quality', axis=1).values
    y = df['quality'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    splits[name] = {'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test}
    print(f"{name}: Train shape {X_train.shape}, Test shape {X_test.shape}")


Red Wine: Train shape (1279, 11), Test shape (320, 11)
White Wine: Train shape (3918, 11), Test shape (980, 11)
Combined Wine: Train shape (5197, 11), Test shape (1300, 11)


In [ ]:
from collections import Counter
from sklearn.metrics import f1_score

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        
    def is_leaf_node(self):
        return self.value is not None

class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=10, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.root = None
        
    def fit(self, X, y):
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._grow_tree(X, y)
        
    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
        
        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)

        best_feature, best_thresh = self._best_split(X, y, feat_idxs)
        
        if best_feature is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        left_idxs, right_idxs = self._split(X[:, best_feature], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth+1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth+1)
        return Node(best_feature, best_thresh, left, right)
        
    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None
        
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thr in thresholds:
                gain = self._information_gain(y, X_column, thr)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr
                    
        return split_idx, split_threshold
        
    def _information_gain(self, y, X_column, threshold):
        parent_entropy = self._entropy(y)
        
        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0
            
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l/n) * e_l + (n_r/n) * e_r
        
        information_gain = parent_entropy - child_entropy
        return information_gain
        
    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs
        
    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log(p) for p in ps if p > 0])
        
    def _most_common_label(self, y):
        if len(y) == 0:
            return 0
        counter = Counter(y)
        return counter.most_common(1)[0][0]
        
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
        
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
            
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

print("--- Decision Tree (NumPy) ---")
for name, data in splits.items():
    clf = DecisionTree(max_depth=10)
    clf.fit(data['X_train'], data['y_train'])
    y_pred = clf.predict(data['X_test'])
    f1 = f1_score(data['y_test'], y_pred, average='weighted')
    print(f"{name} F1 Score: {f1:.4f}")


--- Decision Tree (NumPy) ---
Red Wine F1 Score: 0.5556
White Wine F1 Score: 0.5467
Combined Wine F1 Score: 0.5485


In [ ]:
class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.trees = []
        
    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth,
                                min_samples_split=self.min_samples_split,
                                n_features=self.n_features)
            X_sample, y_sample = self._bootstrap_samples(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
            
    def _bootstrap_samples(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]
        
    def _most_common_label(self, y):
        if len(y) == 0:
            return 0
        counter = Counter(y)
        return counter.most_common(1)[0][0]
        
    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(predictions, 0, 1)
        predictions = np.array([self._most_common_label(preds) for preds in tree_preds])
        return predictions

print("--- Random Forest (NumPy) ---")
for name, data in splits.items():
    n_features = int(np.sqrt(data['X_train'].shape[1]))
    rf = RandomForest(n_trees=10, max_depth=10, n_features=n_features)
    rf.fit(data['X_train'], data['y_train'])
    y_pred = rf.predict(data['X_test'])
    f1 = f1_score(data['y_test'], y_pred, average='weighted')
    print(f"{name} F1 Score: {f1:.4f}")


--- Random Forest (NumPy) ---
Red Wine F1 Score: 0.6109
White Wine F1 Score: 0.5619
Combined Wine F1 Score: 0.5796


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("--- Sklearn Models ---")
for name, data in splits.items():
    print(f"\nDataset: {name}")

    dt_sklearn = DecisionTreeClassifier(max_depth=10, random_state=42)
    dt_sklearn.fit(data['X_train'], data['y_train'])
    y_pred_dt = dt_sklearn.predict(data['X_test'])
    f1_dt = f1_score(data['y_test'], y_pred_dt, average='weighted')
    print(f"Decision Tree F1 Score: {f1_dt:.4f}")

    rf_sklearn = RandomForestClassifier(n_estimators=10, max_depth=10, random_state=42)
    rf_sklearn.fit(data['X_train'], data['y_train'])
    y_pred_rf = rf_sklearn.predict(data['X_test'])
    f1_rf = f1_score(data['y_test'], y_pred_rf, average='weighted')
    print(f"Random Forest F1 Score: {f1_rf:.4f}")


--- Sklearn Models ---

Dataset: Red Wine
Decision Tree F1 Score: 0.5409
Random Forest F1 Score: 0.6119

Dataset: White Wine
Decision Tree F1 Score: 0.5498
Random Forest F1 Score: 0.5578

Dataset: Combined Wine
Decision Tree F1 Score: 0.5454
Random Forest F1 Score: 0.5798
